# Creating Null Models for Novel Domain Architectures

Given the results from the cancer fusion gene analysis, we will establish some null models to determine the effect sizes and if we are observing false discoveries in the data.

**Note** this will take a long time (2+ hrs) to run because of how many trials are being run for the TCGA equivalent size.

In [12]:
import pandas as pd
import numpy as np
import dansy
import random
import fusionClasses as fc
import EnrichementAnalysis

In [13]:
ref_df = dansy.import_proteome_files(ref_file_dir='./data/Current_Human_Proteome',
                                     ref_file_suffix='2026_0715.csv')
exon_information = pd.read_csv('Gene_exon_information.csv', index_col=0)
gene_conv = pd.read_csv('ENSEMBL_Gene_Conversion.csv')
valid_uniprots = list(set(ref_df['UniProt ID']).intersection(gene_conv['UniProtKB/Swiss-Prot ID'].unique()))

In [14]:
proteome_net = dansy.dansy(ref=ref_df, n=10)

Starting to fetch n-grams.
Finished getting all n-grams
Collapsing n-grams
Finished creating edgelist


In [15]:
x = gene_conv.filter(['Gene stable ID', 'Gene name','UniProtKB/Swiss-Prot ID','Chromosome/scaffold name']).drop_duplicates()
conv_dict = x.set_index('UniProtKB/Swiss-Prot ID')['Gene stable ID'].to_dict()
name_conv = x.set_index('UniProtKB/Swiss-Prot ID')['Gene name'].to_dict()
chr_info = x.set_index('UniProtKB/Swiss-Prot ID')['Chromosome/scaffold name'].to_dict()

In [16]:
# Now let's take all the pairs and within each randomly choose exons to include
exons_grouped = exon_information.groupby('Gene stable ID')

In [17]:
# Let's import the enrichment results from the TCGA results to find which natural n-grams to focus on getting FPR values for.
tcga_res= pd.read_csv('TCGA_Fusion_Domain_Enrichment_Results.csv', index_col=0, skiprows=[1], header=0)
ngrams2check = tcga_res.ngram.values.tolist()

In [18]:
# run through each pair and grab a random exon start point to call the breakpoint
natural_ngrams  = set(proteome_net.ngrams).union(proteome_net.collapsed_ngrams)
null_res_list = []
for i in range(500):
    random.seed(i*2)
    n = 15000 # How many pairs to create
    pairs = np.reshape(random.choices(valid_uniprots, k=n*2), (n,2)).tolist()
    null_fusions = []
    for pair in pairs:
        bp_info = pd.Series(index=['h_uniprot', 't_uniprot','h_pos','t_pos', 'h_chr', 't_chr', 'h_gene', 't_gene', 'h_symbol','t_symbol'],
                            dtype=object)
        bp_info['h_uniprot'] = pair[0]
        bp_info['t_uniprot'] = pair[1]
        bp_info['h_pos'] = random.choice(exons_grouped.get_group(conv_dict[pair[0]])['Exon region start (bp)'].values)
        bp_info['t_pos'] = random.choice(exons_grouped.get_group(conv_dict[pair[1]])['Exon region start (bp)'].values)
        bp_info['h_gene'] = conv_dict[pair[0]]
        bp_info['t_gene'] = conv_dict[pair[1]]
        bp_info['h_symbol'] = name_conv[pair[0]]
        bp_info['t_symbol'] = name_conv[pair[1]]
        bp_info['h_chr'] = chr_info[pair[0]]
        bp_info['t_chr'] = chr_info[pair[1]]
        null_fusions.append(bp_info)
        
    null_fusions = pd.DataFrame.from_records(null_fusions)
    null_fusions['name'] = null_fusions.h_symbol.str.cat(null_fusions.t_symbol, sep = '--')
    fusion_dansy = fc.fusionCollection(null_fusions, mapper={k:k for k in null_fusions.columns}, add_cols=[], silent=True)
    fusion_dansy.perform_dansy_analysis(dansy_obj=proteome_net)
    y = fusion_dansy.summarize_dansy_results()
    y['Novel_Architecture'] = y['dansy_impact'].apply(lambda x: str('Novel' in x))

    if i == 0:
        y.to_csv('260818_15K_Random_Pairs_DANSy_Results.csv')

    # Let's grab all the n-grams and all the domains from the natural proteome but limited to 3-grams to save time
    # null_ngrams = [k for k in dansy.ngramUtilities.return_ngrams_from_list(y.domain_architecture.values,3)]
    
    # Now limiting them to only those found in the proteome
    # null_ngrams = list(set(null_ngrams).intersection(natural_ngrams))
    null_enrichment_novel = EnrichementAnalysis.EnrichmentAnalysis(y,ngrams2check, {'category':'Novel_Architecture','domain_architecture':'domain_architecture'})
    null_res_list.append(null_enrichment_novel.res.set_index('ngram').True_q)

100%|██████████| 6467/6467 [00:10<00:00, 592.04it/s]


There were 2138 fusion domain architectures previously found in the proteome.


100%|██████████| 6607/6607 [00:11<00:00, 588.68it/s]


There were 2144 fusion domain architectures previously found in the proteome.


100%|██████████| 6595/6595 [00:11<00:00, 579.65it/s]


...

100%|██████████| 6417/6417 [00:07<00:00, 858.08it/s]


There were 2143 fusion domain architectures previously found in the proteome.


100%|██████████| 6504/6504 [00:08<00:00, 780.45it/s]


There were 2169 fusion domain architectures previously found in the proteome.


100%|██████████| 5166/5166 [00:21<00:00, 244.72it/s]


In [19]:
tcga_null_dists = pd.concat(null_res_list, axis = 1)
tcga_null_dists.to_csv('260820_Null_p_val_dists_15K_tcga_500_trials.csv')

Now repeating but with fewer pairs to recapture the cutoffs for the CCLE datasets instead

In [10]:
# run through each pair and grab a random exon start point to call the breakpoint
natural_ngrams  = set(proteome_net.ngrams).union(proteome_net.collapsed_ngrams)
null_res_list = []
for i in range(500):
    random.seed(i*2)
    n = 3000 # How many pairs to create
    pairs = np.reshape(random.choices(valid_uniprots, k=n*2), (n,2)).tolist()
    null_fusions = []
    for pair in pairs:
        bp_info = pd.Series(index=['h_uniprot', 't_uniprot','h_pos','t_pos', 'h_chr', 't_chr', 'h_gene', 't_gene', 'h_symbol','t_symbol'],
                            dtype=object)
        bp_info['h_uniprot'] = pair[0]
        bp_info['t_uniprot'] = pair[1]
        bp_info['h_pos'] = random.choice(exons_grouped.get_group(conv_dict[pair[0]])['Exon region start (bp)'].values)
        bp_info['t_pos'] = random.choice(exons_grouped.get_group(conv_dict[pair[1]])['Exon region start (bp)'].values)
        bp_info['h_gene'] = conv_dict[pair[0]]
        bp_info['t_gene'] = conv_dict[pair[1]]
        bp_info['h_symbol'] = name_conv[pair[0]]
        bp_info['t_symbol'] = name_conv[pair[1]]
        bp_info['h_chr'] = chr_info[pair[0]]
        bp_info['t_chr'] = chr_info[pair[1]]
        null_fusions.append(bp_info)
        
    null_fusions = pd.DataFrame.from_records(null_fusions)
    null_fusions['name'] = null_fusions.h_symbol.str.cat(null_fusions.t_symbol, sep = '--')
    fusion_dansy = fc.fusionCollection(null_fusions, mapper={k:k for k in null_fusions.columns}, add_cols=[],silent=True)
    fusion_dansy.perform_dansy_analysis(dansy_obj=proteome_net)
    y = fusion_dansy.summarize_dansy_results()
    y['Novel_Architecture'] = y['dansy_impact'].apply(lambda x: str('Novel' in x))
    # Let's grab all the n-grams and all the domains from the natural proteome
    null_ngrams = [k for k in dansy.ngramUtilities.return_ngrams_from_list(y.domain_architecture.values,3)]
    
    # Now limiting them to only those found in the proteome
    null_ngrams = list(set(null_ngrams).intersection(natural_ngrams))
    null_enrichment_novel = EnrichementAnalysis.EnrichmentAnalysis(y,null_ngrams, {'category':'Novel_Architecture','domain_architecture':'domain_architecture'})
    null_res_list.append(null_enrichment_novel.res.set_index('ngram').True_q)

100%|██████████| 1646/1646 [00:02<00:00, 623.81it/s]


There were 691 fusion domain architectures previously found in the proteome.


100%|██████████| 1656/1656 [00:01<00:00, 876.58it/s]


There were 720 fusion domain architectures previously found in the proteome.


100%|██████████| 1661/1661 [00:01<00:00, 885.99it/s]


...

100%|██████████| 1649/1649 [00:02<00:00, 597.55it/s]


There were 710 fusion domain architectures previously found in the proteome.


100%|██████████| 1641/1641 [00:02<00:00, 609.26it/s]


There were 695 fusion domain architectures previously found in the proteome.


100%|██████████| 3555/3555 [00:05<00:00, 702.00it/s]


In [11]:
ccle_null_dists = pd.concat(null_res_list, axis = 1)
ccle_null_dists.to_csv('260820_Null_p_val_dists_3K_ccle.csv')